In [3]:
import pandas as pd
import json
import ollama

/var/folders/_3/wtwzgv1d3rlfz233qkf36kg00000gp/T/ipykernel_21270/3992394387.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [4]:
df_train = pd.read_csv('data/codex/train.txt', sep='\t', header=None, names=['Head','Relation','Tail'])
df_test = pd.read_csv('data/codex/test.txt', sep='\t', header=None, names=['Head','Relation','Tail'])
df_valid = pd.read_csv('data/codex/valid.txt', sep='\t', header=None, names=['Head','Relation','Tail'])

In [5]:
with open('data/codex/entities.json', 'r') as file:
    entities = json.load(file)

with open('data/codex/relations.json', 'r') as file:
    relations = json.load(file)

In [6]:
def id2name_df(id_df:pd.DataFrame, entities_dict:dict, relations_dict:dict)-> pd.DataFrame:
    name_dict = {}
    for id, row in id_df.iterrows():
        # get id
        head_id = row['Head']
        relation_id = row['Relation']
        tail_id = row['Tail']
        # get label out of id
        head = entities_dict[head_id]['label']
        relation = relations_dict[relation_id]['label']
        tail = entities_dict[tail_id]['label']
        name_dict[id] = [head,relation,tail]
    name_df = pd.DataFrame.from_dict(name_dict,orient='index',columns=['Head','Relation','Tail'])
    return name_df

In [7]:
df_train_name = id2name_df(df_train, entities, relations)
df_test_name = id2name_df(df_test, entities, relations)
df_valid_name = id2name_df(df_valid, entities, relations)

In [8]:
df_name = pd.concat([df_train_name, df_test_name, df_valid_name]).reset_index(drop=True)

In [9]:
df_name_sample = df_name.sample(int(0.8*len(df_name)))

In [10]:
from cand_gen.embedding import train_model
from cand_gen import triple_gen
import cand_gen.embedding
import importlib
importlib.reload(cand_gen.embedding)

/opt/homebrew/Caskroom/miniconda/base/envs/kg-emb/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'cand_gen.embedding' (namespace) from ['/Users/fieng/Project/KGC-llm/cand_gen/embedding', '/Users/fieng/Project/KGC-llm/cand_gen/embedding']>

In [11]:
candidates_df = triple_gen.generate_all_candidates(df_name_sample)

In [12]:
# Get rows where 'id' and 'value' in df1 are not in df2
df_missing = df_name[~df_name.apply(tuple, axis=1).isin(df_name_sample.apply(tuple, axis=1))]
df_coverage = candidates_df[candidates_df.apply(tuple, axis=1).isin(df_missing.apply(tuple, axis=1))]
coverage = len(df_coverage) / len(df_missing)

In [13]:
import importlib

In [14]:
import llm_eval.eval
from llm_eval.eval import eval_df, eval_df_context
importlib.reload(llm_eval.eval)

<module 'llm_eval.eval' from '/Users/fieng/Project/KGC-llm/llm_eval/eval.py'>

In [ ]:
score_cand = eval_df(candidates_df, 500)
score_cand_cont = eval_df_context(candidates_df, 500)

In [ ]:
def clean_score(list_score, thresh) -> list[float]:
    list_score = [item.rstrip('.') if item.endswith('.') else item for item in list_score]
    list_score = [item for item in list_score if item and item.strip()]
    score_bin = [1 if float(item) > thresh else 0 for item in list_score]
    return score_bin

In [ ]:
score_cand_clean = clean_score(score_cand, 0.6)
score_cand_cont_clean = clean_score(score_cand_cont, 0.6)

In [16]:
import cand_gen.embedding
from cand_gen.embedding import train_model
importlib.reload(cand_gen.embedding)

<module 'cand_gen.embedding' (namespace) from ['/Users/fieng/Project/KGC-llm/cand_gen/embedding', '/Users/fieng/Project/KGC-llm/cand_gen/embedding']>

In [ ]:

train_df = train_model.create_dataset(
    df_name_sample)
test_df = train_model.create_dataset(
    df_name_sample.sample(n=50))


embedding_dim = 5
model_kwargs = {"embedding_dim": embedding_dim}

model_dict = {}
model_list = ['TransE','TransH','TransF','TransR','TransD']
for model_name in model_list:
    experiment_name = model_name+f"_dim{embedding_dim}"
    model = train_model.create_pipeline(train_df, test_df,
                        model_name, model_kwargs, experiment_name)
    model_dict[model_name] = model

In [19]:
import cand_gen.embedding.get_emb_transe
from cand_gen.embedding.get_emb_transe import get_list_dist

In [20]:
def filter_candidates(candidates_df:pd.DataFrame, threshold:float) -> pd.DataFrame:
    candidates_sample_df = candidates_df[candidates_df['distance']<threshold]
    candidates_sample_df = candidates_sample_df[['Head','Relation','Tail']]
    return candidates_sample_df

def compute_missing_df(original_df:pd.DataFrame, sample_df:pd.DataFrame) -> pd.DataFrame:
    df_missing = original_df[~df_name.apply(tuple, axis=1).isin(sample_df.apply(tuple, axis=1))]
    return df_missing

def compute_coverage(filtred_df :pd.DataFrame, df_missing:pd.DataFrame) -> float:
    df_coverage = filtred_df[filtred_df.apply(tuple, axis=1).isin(df_missing.apply(tuple, axis=1))]
    coverage = len(df_coverage) / len(df_missing)
    return coverage

def compute_cand_completness(filtred_df :pd.DataFrame, df_missing:pd.DataFrame) -> float:
    df_coverage = filtred_df[filtred_df.apply(tuple, axis=1).isin(df_missing.apply(tuple, axis=1))]
    coverage = len(df_coverage) / len(df_missing)
    return coverage

In [25]:
model = model_dict['TransH']
list_dist = get_list_dist(candidates_df, model.model, train_df)
candidates_df['distance'] = list_dist


In [21]:
transh_score_dict = {}

model = model_dict['TransH']
list_dist = get_list_dist(candidates_df, model.model, train_df)
candidates_df['distance'] = list_dist

mean_dist = candidates_df['distance'].mean()
std_dist = candidates_df['distance'].std()

threshold_list = [mean_dist- std_dist, mean_dist-0.5*std_dist,mean_dist,
                mean_dist+0.5*std_dist,mean_dist + std_dist]
for threshold in threshold_list:
    filtred_df = filter_candidates(candidates_df, threshold)
    df_missing = compute_missing_df(df_name, df_name_sample)
    coverage = compute_coverage(filtred_df, df_missing)
    reduction_ratio = len(filtred_df)/len(candidates_df)
    candidates_completness = 
    new_score = coverage/reduction_ratio
    transh_score_dict[threshold] = new_score, reduction_ratio


SyntaxError: invalid syntax (3820061775.py, line 17)

In [71]:
transe_score_dict = {}

model = model_dict['TransE']
list_dist = get_list_dist(candidates_df, model.model, train_df)
candidates_df['distance'] = list_dist
proportion_dict = {}

mean_dist = candidates_df['distance'].mean()
std_dist = candidates_df['distance'].std()

threshold_list = [mean_dist- std_dist, mean_dist-0.5*std_dist,mean_dist,
                mean_dist+0.5*std_dist,mean_dist + std_dist]
for threshold in threshold_list:
    filtred_df = filter_candidates(candidates_df, threshold)
    df_missing = compute_missing_df(df_name, df_name_sample)
    coverage = compute_coverage(filtred_df, df_missing)
    reduction_ratio = len(filtred_df)/len(candidates_df)
    new_score = coverage/reduction_ratio
    transe_score_dict[threshold] = new_score, reduction_ratio
    proportion_dict[threshold] = reduction_ratio

In [79]:
transf_score_dict = {}

model = model_dict['TransF']
list_dist = get_list_dist(candidates_df, model.model, train_df)
candidates_df['distance'] = list_dist

mean_dist = candidates_df['distance'].mean()
std_dist = candidates_df['distance'].std()

threshold_list = [mean_dist- std_dist, mean_dist-0.5*std_dist,mean_dist,
                mean_dist+0.5*std_dist,mean_dist + std_dist]
for threshold in threshold_list:
    filtred_df = filter_candidates(candidates_df, threshold)
    df_missing = compute_missing_df(df_name, df_name_sample)
    coverage = compute_coverage(filtred_df, df_missing)
    reduction_ratio = len(filtred_df)/len(candidates_df)
    new_score = coverage/reduction_ratio
    transf_score_dict[threshold] = new_score, reduction_ratio


In [80]:
transd_score_dict = {}

model = model_dict['TransD']
list_dist = get_list_dist(candidates_df, model.model, train_df)
candidates_df['distance'] = list_dist

mean_dist = candidates_df['distance'].mean()
std_dist = candidates_df['distance'].std()

threshold_list = [mean_dist- std_dist, mean_dist-0.5*std_dist,mean_dist,
                mean_dist+0.5*std_dist,mean_dist + std_dist]
for threshold in threshold_list:
    filtred_df = filter_candidates(candidates_df, threshold)
    df_missing = compute_missing_df(df_name, df_name_sample)
    coverage = compute_coverage(filtred_df, df_missing)
    reduction_ratio = len(filtred_df)/len(candidates_df)
    new_score = coverage/reduction_ratio
    transd_score_dict[threshold] = new_score, reduction_ratio


In [55]:
random_score_dict =  {}
for proportion in proportion_dict.values():
    sample_size = int(proportion * len(candidates_df))
    filtred_df = candidates_df.sample(sample_size)
    filtred_df = filtred_df[['Head','Relation','Tail']]
    df_missing = compute_missing_df(df_name, df_name_sample)
    reduction_ratio = len(filtred_df)/len(candidates_df)
    new_score = coverage/reduction_ratio
    random_score_dict[proportion] = new_score, reduction_ratio

In [2]:
missing_df = compute_missing_df(df_name, df_name_sample)
cand_df = candidates_df[['Head','Relation','Tail']]
true_cand_df = cand_df[cand_df.apply(tuple, axis=1).isin(missing_df.apply(tuple, axis=1))]
false_cand_df = cand_df[~cand_df.apply(tuple, axis=1).isin(missing_df.apply(tuple, axis=1))]

NameError: name 'compute_missing_df' is not defined

In [1]:
true_cand_df['Missing'] = 1
false_cand_df['Missing'] = 0
test_df = pd.concat([true_cand_df, false_cand_df])

NameError: name 'true_cand_df' is not defined

In [ ]:
cand